# Strand C (instrument): do HTTP Archive and Blacklight agree when contemporaneous?

**Purpose:** strands A and B lean on HTTP Archive request maps. This notebook
asks how those measures relate to Blacklight's when both instruments observe
the same domains at (nearly) the same time — the Jan-2025 HTTP Archive crawl
vs the ~Jan-2025 Blacklight scans, mobile emulation on both sides (Blacklight
emulated an iPhone 13 Mini).

Some constructions differ by design and the labels say so: HTTP Archive sees
only cookies set via response headers (not JS-set), and flags any
facebook/GA-GTM request, while Blacklight tests specifically for the FB pixel
and GA *remarketing*. HA also unions requests across every crawled origin of
a registered domain, while Blacklight loads one page. Presence agreement and
rank correlations absorb scale; construct gaps remain and are part of the
finding.

In [1]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))

import config
from utilities import pandas_to_tex, save_mpl_fig


Checking that all paths exist:
{'web_mobile': False, 'web_desktop': False, 'web': False, 'yg_profile': False, 'blacklight': False, 'who': False}


In [2]:
m = pd.read_csv(config.FP_HA_DOMAIN_MEASURES)
bl = pd.read_csv(config.FP_BLACKLIGHT)
ha_jan = m[(m.crawl == "blacklight_match") & (m.client == "mobile")].copy()
ha_jan["filename"] = ha_jan["private_domain"].str.replace(".", "_", regex=False)
j = ha_jan.merge(bl, on="filename", suffixes=("_ha", "_bl"))
print(f"domains measured by both instruments: {len(j):,}")

domains measured by both instruments: 6,902


In [3]:
PAIRS = [
    ("ddg_join_ads_ha", "ddg_join_ads_bl", "Ad trackers (same construct)"),
    ("ddg_known_trackers", "ddg_join_ads_bl", "Any known tracker (HA) vs ad trackers (BL)"),
    ("third_party_cookies_ha", "third_party_cookies_bl", "3p cookies: header-set (HA) vs all (BL)"),
    ("fb_pixel_ha", "fb_pixel_bl", "FB requests (HA) vs FB pixel (BL)"),
    ("google_analytics_ha", "google_analytics_bl", "GA/GTM presence (HA) vs GA remarketing (BL)"),
]

rows = []
for ha_col, bl_col, label in PAIRS:
    pa, pb = (j[ha_col] > 0), (j[bl_col] > 0)
    rows.append({
        "measure": label,
        "n_domains": len(j),
        "ha_prev_pct": 100 * pa.mean(),
        "bl_prev_pct": 100 * pb.mean(),
        "agree_pct": 100 * (pa == pb).mean(),
        "spearman_counts": j[ha_col].corr(j[bl_col], method="spearman"),
    })
agree = pd.DataFrame(rows)
agree.round(2)

,measure,n_domains,ha_prev_pct,bl_prev_pct,agree_pct,spearman_counts
0,Ad trackers (same construct),6902,92.90,82.61,86.15,0.60
1,Any known tracker (HA) vs ad trackers (BL),6902,99.19,82.61,83.08,0.54
2,3p cookies: header-set (HA) vs all (BL),6902,30.21,65.85,49.15,0.19
3,FB requests (HA) vs FB pixel (BL),6902,44.80,27.48,77.96,0.58
4,GA/GTM presence (HA) vs GA remarketing (BL),6902,86.25,4.35,17.81,0.06


In [4]:
tex = agree.copy()
tex["n_domains"] = tex["n_domains"].map("{:,}".format)
for c in ["ha_prev_pct", "bl_prev_pct", "agree_pct"]:
    tex[c] = tex[c].map("{:.1f}".format)
tex["spearman_counts"] = tex["spearman_counts"].map("{:.2f}".format)
pandas_to_tex(tex, os.path.join(config.TABLES_DIR, "ha_bl_agreement"))
print("wrote tables/ha_bl_agreement.tex")

wrote tables/ha_bl_agreement.tex


## Reading the results

High agreement on a measure means the instruments are near-interchangeable
there, so strand A's temporal comparisons and strand B's calibrated fills rest
on solid ground for that measure. Low agreement on the construct-mismatched
rows (cookies, GA) is expected and bounds how literally those HA measures
should be read against Blacklight's — the calibration in `07_coverage_bounds`
conditions on exactly this joint distribution rather than assuming the
constructs coincide.